# ☀️ Exercise 2 — Forecasting the output of a solar plant in Malta
**Energy Regulation, Economics and Sustainable Energy Systems in the Mediterranean and MENA Region** · Hands-on session · BALAB, Bari

---
## 🎯 Your mission
You work for the **energy regulator of Malta**. A (fictional) solar plant of **12 MWp** (maximum AC output **8.5 MW**) must send the grid operator a **day-ahead forecast** of its hourly production. A bad forecast means expensive balancing for the grid, so you must build a model that predicts the power (MW) **from the weather forecast**.

You have **18 months** of hourly data (Jan 2023 → Jun 2024). The **last 6 months** (Jul → Dec 2024) are hidden: your instructor will give them to you at the end, to see how good your model really is.

You already did this with **demand** in Exercise 1. Now **you** drive. Same steps, a new problem, and a new model.

---
## 🆕 What is different from Exercise 1
| | Exercise 1 (demand) | Exercise 2 (solar) |
|---|---|---|
| Target | electricity demand | **PV power**: 0 at night, peak at noon |
| Model | Random Forest | **XGBoost** (a different way of combining trees) |
| Zeros | a zero was an error | **zero at night is REAL**, but zero at noon is an error! |
| Error measure | MAPE | MAPE breaks (dividing by 0!), so we use **nMAE** (error as % of the plant size) and MAPE **only in daylight** |
| Who writes the code | the instructor | **you** (with ChatGPT / Claude) |

📘 **XGBoost in one sentence:** like Random Forest it uses many small trees, but the trees are built **one after the other** and every new tree tries to **fix the mistakes of the previous ones** (*boosting*). Random Forest = trees in parallel, then average (*bagging*).

---
## 🤖 You may (and should) use ChatGPT or Claude (free versions are fine)
1. Open ChatGPT or Claude in another tab.
2. **Paste the "context prompt" below once, at the beginning of the chat.**
3. For each step, paste the **step prompt** that we provide (or write your own) and copy the code into a cell.
4. If you get an error: **paste the error message into the chat** and say "fix it". Normal!
5. ⚠️ **Never trust the AI blindly.** Always run the check cells and look at your plots.

### 📋 Context prompt (copy ALL of it into ChatGPT / Claude, once)
```
I am a beginner in Python and I work in Google Colab. I am an energy regulator / economist.
Please write SIMPLE code: no functions, no classes, short lines, and a comment on every important line.
Explain in one or two sentences what the code does. Use pandas, numpy, matplotlib, shap and xgboost only.

THE PROJECT
A fictional solar power plant in Malta: 12 MWp of panels, 8.5 MW maximum AC output. We want to predict the
hourly power produced (MW) from the weather forecast of the same hour. Training data: Jan 2023 - Jun 2024.
Test data (hidden): Jul - Dec 2024.

MY TABLE IS CALLED "df" AND HAS THESE COLUMNS (one row per hour)
timestamp (text like 2023-01-01 14:00), month (1-12), day_of_week (0=Monday), is_holiday (0/1),
solar_elevation_deg (sun height above the horizon, negative = night), ghi_forecast_wm2 (forecast sun power),
cloud_cover_forecast_pct (forecast, 0-100), temperature_c, wind_speed_ms,
pv_power_lag_24h_mw (power of the same hour yesterday), pv_power_mw (TARGET: power produced, MW).

IMPORTANT FACTS
- At night the power is really 0 (tiny negative values like -0.02 are fine). Zeros at night are NOT errors.
- The data has errors: duplicates, missing values, impossible values, and sensor problems.
- Never mix the time order: validation must be the LAST months of the training data.
- Do not use random splits.

Now I will ask you one step at a time. Answer only the step I ask. Give me code I can paste in one Colab cell.
```

## ⚠️ Six things to know before you start
1. **Run cells in order**, from top to bottom. If something is strange: *Runtime → Restart session*, then run all again from the top.
2. **Use these names** in your code, so the check cells work: `df` (your table), `train_part`, `val_part`, `feature_cols`, `model`.
3. **Night is zero.** In a solar plant, 0 MW at night is **normal**. Do not delete those rows!
4. **Time order matters.** Training = the past, validation = the most recent months. **Never** split randomly.
5. **Not every strange value is an error, and not every error looks strange.** Plot first, then decide.
6. If the AI writes a **function** (`def ...`), ask: *"rewrite without functions, in the simplest way"*.

---
## 🗺️ The steps (about 75 minutes)
| # | Step | Time |
|---|---|---|
| 1 | Look at the table | 5 min |
| 2 | Sort in time, remove duplicates | 5 min |
| 3 | Plots: find the problems | 10 min |
| 4 | Clean outliers, fill the gaps | 15 min |
| 5 | Split train / validation | 5 min |
| 6 | Train XGBoost, measure the error | 10 min |
| 7 | SHAP: explain the model | 10 min |
| 8 | Save the final model | 5 min |
| 9 | Test + errors (instructor gives the signal) | 5 min |

## 🔧 Setup and data loading (already written — just run)

In [ ]:
# ===== SETUP: run this cell first. Do not change it. =====
# 1) Install the SHAP library (explainable AI). XGBoost is already installed in Colab.
%pip install -q shap

# 2) Import the libraries ("toolboxes") we will use
import os, random, json, warnings
import numpy as np                      # maths
import pandas as pd                     # tables (like Excel, but in Python)
import matplotlib.pyplot as plt         # plots
import joblib                           # to save the trained model
import shap                             # explainable AI
from xgboost import XGBRegressor        # the machine learning model (NEW: it was Random Forest in exercise 1)
from sklearn.metrics import mean_absolute_error, mean_squared_error   # error measures

warnings.filterwarnings("ignore")       # hide non-critical warnings

# 3) Fix the random seed: everybody obtains exactly the same results
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# 4) Plot size
plt.rcParams["figure.figsize"] = (10, 4)
print("Setup done.")

In [ ]:
# ===== LOAD THE TRAINING DATA (given: do not change) =====
# The file lives in the instructor's public GitHub repository. pandas can read it directly from the link.
TRAIN_URL = "https://raw.githubusercontent.com/mariocaruso96/MENA/main/pv_train.csv"
pv_raw = pd.read_csv(TRAIN_URL)

print("Rows and columns:", pv_raw.shape)
pv_raw.head()

**The columns**

| Column | Meaning |
|---|---|
| `timestamp` | date and hour (one row per hour) |
| `month`, `day_of_week`, `is_holiday` | calendar information |
| `solar_elevation_deg` | height of the sun above the horizon (negative = night) |
| `ghi_forecast_wm2` | **forecast** of the sun's power on the ground (W/m²) |
| `cloud_cover_forecast_pct` | **forecast** cloud cover (%) |
| `temperature_c` | temperature (°C) |
| `wind_speed_ms` | wind speed (m/s) |
| `pv_power_lag_24h_mw` | power of the **same hour yesterday** (known when we make the forecast) |
| **`pv_power_mw`** | **🎯 TARGET: power produced (MW)** |

---
# Step 1 — Look at the table  *(5 min)*
**What to do:** find out how many rows and columns there are, the type of each column, how many values are missing, and the minimum / maximum of each column.

💡 **Hint:** in pandas the table is a *DataFrame*. Useful commands: `.shape`, `.dtypes`, `.isna().sum()`, `.describe()`. The table is called `pv_raw`.

🤖 **Prompt for the AI**
```
Using my table "pv_raw", write simple code that prints: number of rows and columns, the type of each column,
the number of missing values in each column, and min, mean and max of each column.
```
**Think about it:** do you see any **minimum or maximum that is impossible** for a 8.5 MW plant in Malta?

In [ ]:
# Write (or paste) your code here

---
# Step 2 — Sort in time and remove duplicates  *(5 min)*
📘 *Data quality: "garbage in, garbage out".* A time series must be **in order**, with **one row per hour**.

**What to do:**
1. Make a **copy** of `pv_raw` called `df`.
2. Convert `timestamp` into a real date (`pd.to_datetime`).
3. **Sort** by `timestamp`, **drop duplicated hours** (keep the first), and renumber the rows.
4. Create a new column `hour` (0–23) from the timestamp: the hour of the day is the most important clue for solar power.
5. Print how many rows you removed.

💡 **Hint:** `.sort_values(...)`, `.drop_duplicates(subset="timestamp", keep="first")`, `.reset_index(drop=True)`, `df["timestamp"].dt.hour`.

🤖 **Prompt for the AI**
```
Make a copy of pv_raw called df. Convert the column timestamp to datetime, sort by timestamp, drop duplicated timestamps
keeping the first, reset the index. Add a column "hour" with the hour of the day. Print how many rows were removed.
```
**Think about it:** the duplicates may not be identical copies: two rows with the same hour but **different values**. Which one do you keep? (Keeping the first is a simple, documented choice.)

In [ ]:
# Write (or paste) your code here

---
# Step 3 — Plots: find the problems  *(10 min)*
📘 *Always plot before you clean.* Plots show the problems that tables hide.

**What to do — make (at least) these plots:**
1. **Histograms** of `pv_power_mw`, `temperature_c`, `ghi_forecast_wm2`, `cloud_cover_forecast_pct`
2. `temperature_c` **through time** (a line plot)
3. **Scatter plot** `ghi_forecast_wm2` (x) vs `pv_power_mw` (y). It should look like a **line**: more sun = more power.
4. **Average power for each hour of the day** (a bar plot)
5. **One week** of power, hour by hour (for example June 2023)

💡 **Hint:** `plt.hist(...)`, `plt.plot(...)`, `plt.scatter(..., s=3, alpha=0.3)`, `df.groupby("hour")["pv_power_mw"].mean().plot(kind="bar")`. Limit the axes if a few huge values squash the plot (`plt.xlim`, `plt.ylim`).

🤖 **Prompt for the AI**
```
With matplotlib and my table df, draw: (1) four histograms (pv_power_mw, temperature_c, ghi_forecast_wm2,
cloud_cover_forecast_pct) in a 2x2 figure; (2) temperature_c through time; (3) a scatter plot of ghi_forecast_wm2
against pv_power_mw with small transparent points; (4) average pv_power_mw for each hour; (5) pv_power_mw for the week
from 2023-06-12 to 2023-06-18. Simple code, with titles.
```
**📝 Write down the problems you see.** There are **at least 8 different kinds**. Hints on where to look:
- 🌡️ something odd with the **temperature** during some weeks of summer 2023 (units?)
- ☁️ cloud cover cannot be above 100 %…
- ☀️ the sun cannot deliver 4000 W/m²…
- ⚡ the plant cannot produce 40 MW, or **−5 MW**
- 🕛 **zero power at midday**, with a bright sun (inverter off?)
- 📏 a **flat horizontal line** in the power for many hours (frozen sensor?)
- 🕳️ **missing values** everywhere

In [ ]:
# Write (or paste) your code here

---
# Step 4 — Clean: outliers and gaps  *(15 min)*
📘 *Remember Exercise 1:* simple rules, one problem at a time, written down. **Do not delete everything that looks odd.**

**What to do — one rule for each problem you found:**
| Problem | Suggested rule |
|---|---|
| Temperature in °F | values above 50 → convert: `(F − 32) × 5 / 9` |
| Cloud cover = 999 | above 100 → make it missing (`np.nan`) |
| Irradiance impossible | above 1400 W/m² → make it missing |
| Power too high | above 8.6 MW → make it missing |
| Power very negative | below −0.1 MW → make it missing |
| Frozen sensor | the same value 3 hours in a row (and not 0, and not the 8.5 limit) → missing |
| Zero in full sun | power = 0 **and** `solar_elevation_deg` > 15 **and** `ghi_forecast_wm2` > 200 → missing |
| Tiny negatives at night (−0.02) | **not an error**: set them to exactly 0 |
| Gaps in the **inputs** | fill with `.interpolate()` |
| Gaps in the **target** | **never invent** the target: remove those rows (`dropna`) |

💡 **Hints**
- To change some rows: `df.loc[condition, "column"] = new_value`
- A condition looks like `df["temperature_c"] > 50`. Combine with `&` (and) / `|` (or), always **with brackets**.
- "Same as the previous hour": `p.diff() == 0`. "Same for 3 hours": `(p.diff() == 0) & (p.diff(2) == 0)`.
- Do the rules **in order**, and **count** how many rows each rule touches (`.sum()` of the condition).

🤖 **Prompt for the AI**
```
On my table df, apply these cleaning rules one by one, printing how many rows each rule affects:
1) temperature_c > 50 is Fahrenheit: convert to Celsius.
2) cloud_cover_forecast_pct > 100: set to NaN.
3) ghi_forecast_wm2 > 1400: set to NaN.
4) in pv_power_mw: set to NaN if > 8.6, or < -0.1, or if the same value repeats for 3 hours in a row (and the value is between 0.1 and 8.4),
   or if it is exactly 0 while solar_elevation_deg > 15 and ghi_forecast_wm2 > 200.
5) pv_power_mw between -0.1 and 0 (night noise): set to 0.
6) interpolate temperature_c, cloud_cover_forecast_pct and ghi_forecast_wm2.
7) drop the rows where pv_power_mw or pv_power_lag_24h_mw is missing, and reset the index.
Simple code, no functions.
```
⚠️ **Trap:** if the AI suggests *"remove all the rows where power is 0 or negative"*, **say no**: you would delete every night and the model would never learn that the plant sleeps at night.

In [ ]:
# Write (or paste) your code here

**✅ Check (given):** run the cell below. It must tell you the cleaning worked.

In [ ]:
# CHECK (given): did the cleaning work? Everything below should look reasonable.
print("Missing values left:", int(df.isna().sum().sum()), "(must be 0)")
print()
check = df[["temperature_c", "cloud_cover_forecast_pct", "ghi_forecast_wm2", "pv_power_mw"]].agg(["min", "max"])
print(check)
print()
print("Expected: temperature 4-45, cloud 0-100, irradiance below ~1100, power between 0 and 8.5.")

---
# Step 5 — Split in training and validation  *(5 min)*
📘 *Train / validation / test.* The model learns on **training**, we check it on **validation** (data it has not studied), and the **test** (hidden) is the final exam.

**What to do:**
1. Training = all hours **before 1 April 2024**. Validation = **from 1 April 2024**. (In time order! 15 months to learn, 3 months to check.)
2. Make a list `feature_cols` with all the columns **except** `timestamp` and `pv_power_mw`.
3. Create `X_train`, `y_train`, `X_val`, `y_val` (X = the features, y = the target).

💡 **Hint:** `train_part = df[df["timestamp"] < "2024-04-01"]`. The list of features: `[c for c in df.columns if c not in [...]]`.

🤖 **Prompt for the AI**
```
Split my table df by time: train_part = rows with timestamp before 2024-04-01, val_part = rows from 2024-04-01.
Create feature_cols = all columns except timestamp and pv_power_mw. Create X_train, y_train, X_val, y_val. Print the sizes.
```
**Think about it:** why is a random split a bad idea for a time series? (Hint: yesterday's data is almost identical to today's.)

In [ ]:
# Write (or paste) your code here

---
# Step 6 — Train XGBoost and measure the error  *(10 min)*
📘 *Overfitting, baseline, MAE / RMSE.* A model must **beat a baseline**. Here the baseline is *"the power will be the same as yesterday at this hour"* (the column `pv_power_lag_24h_mw`).

**What to do:**
1. Train an `XGBRegressor` on `X_train`, `y_train`. Call it `model`. Use `random_state=SEED` and `n_jobs=1`.
2. Predict on `X_val`. Force the predictions to stay between **0 and 8.5 MW** (`np.clip`).
3. Compute **MAE**, **RMSE**, **nMAE = MAE / 8.5 × 100 (%)**, and **MAPE only in daylight** (rows where real power > 1 MW).
4. Compute the same errors for the **baseline**. Does XGBoost win?
5. Plot **five days** of validation: real, predicted, baseline.

💡 **Hint:** `XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=1)`. Then `.fit(...)` and `.predict(...)`. Errors: `mean_absolute_error`, `np.sqrt(mean_squared_error(...))`.

🤖 **Prompt for the AI**
```
Train an XGBRegressor (n_estimators=300, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
random_state=SEED, n_jobs=1) on X_train, y_train. Predict on X_val and clip the predictions between 0 and 8.5.
Compute and print MAE, RMSE, nMAE (MAE divided by 8.5, in %) and MAPE computed only where the real power is above 1 MW.
Compute the same for the baseline: the column pv_power_lag_24h_mw of val_part. Print a small table with both. Then plot five days
(2024-05-20 to 2024-05-24) of real, predicted and baseline power. Simple code, no functions.
```
**Think about it:**
- Does the model beat the baseline? By how much?
- Why would the plain MAPE be a terrible measure here? (What is 1 ÷ 0?)
- Is an error of 1–2 % of the plant size acceptable for the grid operator?

In [ ]:
# Write (or paste) your code here

---
# Step 7 — SHAP: what does the model look at?  *(10 min)*
📘 *Explainable AI.* SHAP tells how much **each feature pushed each prediction up or down**.

**What to do:**
1. Create a `shap.TreeExplainer` for `model`.
2. Take a **sample of 800 validation rows** (it is faster) and compute the SHAP values.
3. Draw the **summary plot** (the colourful one) and, if you want, the **bar** version.
4. Draw the **dependence plot** of `ghi_forecast_wm2`, coloured by `temperature_c`.

💡 **Hint:** `X_val.sample(800, random_state=SEED)`, `explainer.shap_values(X_sample)`, `shap.summary_plot(shap_values, X_sample)`, `shap.dependence_plot("ghi_forecast_wm2", shap_values, X_sample, interaction_index="temperature_c")`.

🤖 **Prompt for the AI**
```
I have a trained XGBoost model called "model" and a validation table X_val. Use SHAP: create a TreeExplainer, take 800 random rows
of X_val (random_state=42) as X_sample, compute shap_values, and draw (1) the summary plot, (2) the bar summary plot,
(3) the dependence plot of "ghi_forecast_wm2" coloured by "temperature_c". Simple code.
```
**Think about it (this is where you become the expert):**
- Which feature is the **most important**? Does it make physical sense?
- In the summary plot, **red dots on the right** mean *high value → higher power*. Is that true for irradiance? And for **temperature**? (Physics: hot panels are *less* efficient.)
- Are there features that **do nothing** (SHAP ≈ 0)? Would you remove them?
- ⚠️ 📘 *Explaining is not proving causality.* SHAP describes how **the model** reasons, not how the world works.

In [ ]:
# Write (or paste) your code here

---
# Step 8 — Save the final model  *(5 min)*
📘 *Deployment.* A model that is not saved cannot be used tomorrow.

**What to do:**
1. (Good practice) **Retrain** a new model `model_final` with the **same settings** on **all** the clean data (training **and** validation), because the most recent months are the most useful.
2. Save it as **`pv_model.joblib`** (`joblib.dump`).
3. Save the list of feature names as **`pv_feature_cols.json`** (`json.dump`).

⚠️ **Use exactly these two file names**: the final cell will look for them.

💡 **Hint:** `joblib.dump(model_final, "pv_model.joblib")` and `with open("pv_feature_cols.json", "w") as f: json.dump(feature_cols, f)`.

🤖 **Prompt for the AI**
```
Retrain my XGBRegressor (same parameters, random_state=SEED, n_jobs=1) on all rows of df: X_all = df[feature_cols], y_all = df["pv_power_mw"],
and call it model_final. Save it with joblib to "pv_model.joblib" and save the list feature_cols to "pv_feature_cols.json" with json.
```

In [ ]:
# Write (or paste) your code here

---
# Step 9 — The final exam: the test  *(5 min)*
📘 *Inference on completely new data.* The test file (**1 July → 31 December 2024**) has **never** been seen by your model. Your instructor tells you when the file is available.

**What to do:**
1. Load the saved model and the feature list.
2. Load the test file from the link given by your instructor (`TEST_URL`). It has the same columns as the training file, **including the real power**.
3. Prepare it **exactly like the training data** (convert the date, sort, create `hour`).
4. Predict (columns in the **same order** as training!), clip between 0 and 8.5 MW.
5. Compute **MAE, RMSE, nMAE and daylight MAPE**, and the same for the baseline.
6. Plot a week of real vs predicted.

💡 **Hint:** this is a copy of Step 6, with the test table instead of `val_part`. Set `model_loaded.set_params(n_jobs=1)` after loading, so that everybody gets the same numbers.

🤖 **Prompt for the AI**
```
Load my model with joblib ("pv_model.joblib") and the feature names from "pv_feature_cols.json". Read the test table with
pd.read_csv(TEST_URL), convert timestamp to datetime, sort by timestamp, and create the column "hour". Predict pv_power_mw using the columns in
feature_cols_loaded, clip between 0 and 8.5. Compute MAE, RMSE, nMAE (% of 8.5 MW), and MAPE only where real power is above 1 MW.
Compute the same for the baseline column pv_power_lag_24h_mw. Print a table. Plot one week (2024-08-05 to 2024-08-11): real vs predicted. Simple code.
```

In [ ]:
TEST_URL = "PASTE THE LINK GIVEN BY THE INSTRUCTOR HERE"

# Write (or paste) your code here

---
## 🎤 Your 4 slides
Use the AI to help you write and format the slides, but **the content must be yours**.

| Slide | Content | Questions to answer |
|---|---|---|
| **1. Data and problems** | what you found and how you fixed it | Which problems did you find? Which did you **not** touch, and why (think: night zeros)? |
| **2. Model and validation** | how you split, the error vs the baseline | Does XGBoost beat the baseline? By how much? |
| **3. What SHAP says** | the plots of Step 7 | What does the model look at? Does it match the physics? |
| **4. Verdict** | the test result | Does the test **confirm** your validation? Would you trust it for a grid decision? **What would you monitor?** |

**Hint for slide 4:** compare the test error with the validation error. The test covers **July–December**, the validation only **April–June**. What could be different?